[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S11/S11_02_transfer_learning.ipynb)

# S11 · Práctica 2 — *Transfer learning* y *fine-tuning*

**Tiempo estimado:** 1 h 45 min · **Sprint 11** (UD 4.5)

**Contexto TurisData.** Entrenar una CNN desde cero necesita miles de fotos etiquetadas, y el cliente tiene
solo unos cientos. La solución habitual es **reutilizar una red ya entrenada** en otra tarea y adaptarla:
es el *transfer learning*.

**Al terminar sabrás:**
1. Qué se reutiliza (la "base" convolucional) y qué se reentrena (la "cabeza" clasificadora).
2. Congelar capas, entrenar una cabeza nueva y hacer *fine-tuning* de las últimas capas.
3. Medir cuánto ayuda el *transfer learning* cuando hay pocos datos.
4. Cómo se haría con pesos reales (MobileNetV2 con ImageNet) y qué cambia.

**Dos ramas.**
- **Rama offline (ejecutable en el taller):** como no hay internet ni pesos de ImageNet, el propio notebook
  **preentrena una red pequeña** en una tarea auxiliar sintética (formas y texturas) y después la reutiliza para reconocer
  tipos de alojamiento **esquemáticos** con pocas imágenes. Demuestra la **mecánica** completa.
- **Rama Colab (requiere internet, sección 6):** carga de `MobileNetV2` con pesos de ImageNet. **No verificada** en el taller.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import functools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

## 0. Generador de imágenes sintéticas
Estas funciones dibujan con PIL (i) **formas y texturas** (para el preentrenamiento) y (ii) **alojamientos
esquemáticos** (la tarea real). No hace falta que las estudies: ejecútala y sigue. Son dibujos, **no fotos reales**.

In [ ]:
from PIL import Image, ImageDraw

CLASES_ALOJ = ["hotel", "apartamentos", "casa_rural", "camping", "piscina"]
CLASES_FORMAS = ["rayas_horizontales", "rayas_verticales", "diagonales", "cuadricula", "puntos", "circulo", "rectangulo", "triangulo"]


def _color(rng, base, var=30):
    """Color aleatorio cercano a `base` (tupla RGB)."""
    return tuple(int(np.clip(c + rng.integers(-var, var + 1), 0, 255)) for c in base)


def _final(img, rng, ruido=8):
    """Convierte a uint8 con un poco de ruido y variación de brillo."""
    a = np.asarray(img, dtype="float32") * rng.uniform(0.85, 1.15)
    a = a + rng.normal(0, ruido, a.shape)
    return np.clip(a, 0, 255).astype("uint8")


def _ventanas(d, x0, y0, x1, y1, filas, cols, color, margen):
    """Cuadrícula de ventanas dentro del rectángulo (x0, y0, x1, y1)."""
    ancho, alto = (x1 - x0) / cols, (y1 - y0) / filas
    for f in range(filas):
        for c in range(cols):
            a, b = x0 + c * ancho + margen * ancho, y0 + f * alto + margen * alto
            d.rectangle([a, b, a + ancho * (1 - 2 * margen), b + alto * (1 - 2 * margen)], fill=color)


def dibujar_alojamiento(clase, rng, tam=48, ruido=8):
    """Dibujo esquemático (tam×tam×3, uint8) de un tipo de alojamiento. Datos SINTÉTICOS, no fotos reales."""
    E = 3                                   # supermuestreo para suavizar bordes
    S = tam * E
    img = Image.new("RGB", (S, S), _color(rng, (170, 205, 235), 25))          # cielo
    d = ImageDraw.Draw(img)
    if rng.random() < 0.5:                                                    # sol opcional
        cx, cy, r = rng.uniform(0.1, 0.9) * S, rng.uniform(0.08, 0.25) * S, rng.uniform(0.05, 0.08) * S
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 235, 120))
    hor = rng.uniform(0.62, 0.72) * S                                         # línea del horizonte
    verde = _color(rng, (120, 170, 90), 25)
    if clase == "piscina":
        d.rectangle([0, hor * 0.6, S, S], fill=_color(rng, (225, 210, 170), 15))   # terraza
        x0, x1 = rng.uniform(0.08, 0.2) * S, rng.uniform(0.75, 0.92) * S
        y0, y1 = rng.uniform(0.5, 0.58) * S, rng.uniform(0.82, 0.94) * S
        d.rectangle([x0 - 3, y0 - 3, x1 + 3, y1 + 3], fill=(245, 245, 245))
        d.rectangle([x0, y0, x1, y1], fill=_color(rng, (60, 150, 210), 20))
        for k in range(3):
            yy = y0 + (k + 1) * (y1 - y0) / 4
            d.line([x0 + 6, yy, x1 - 6, yy], fill=(150, 210, 240), width=2)
        for k in range(2):                                                    # tumbonas
            tx = rng.uniform(0.1, 0.8) * S
            d.rectangle([tx, hor * 0.72, tx + 0.1 * S, hor * 0.72 + 0.04 * S], fill=_color(rng, (240, 120, 60), 30))
    else:
        d.rectangle([0, hor, S, S], fill=verde)                                # césped
    if clase == "hotel":
        w, h = rng.uniform(0.30, 0.42) * S, rng.uniform(0.5, 0.66) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (220, 200, 170)))
        _ventanas(d, x0, y0 + 0.02 * S, x0 + w, hor, int(rng.integers(4, 6)), 3, (60, 90, 140), 0.22)
    elif clase == "apartamentos":
        w, h = rng.uniform(0.7, 0.86) * S, rng.uniform(0.28, 0.38) * S
        x0 = rng.uniform(0.02 * S, 0.98 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 205)))
        _ventanas(d, x0, y0, x0 + w, hor + 0.05 * S, 2, int(rng.integers(5, 7)), (70, 100, 150), 0.25)
        for k in (1, 2):                                                       # balcones (líneas horizontales)
            yy = y0 + k * (hor + 0.05 * S - y0) / 2
            d.line([x0, yy, x0 + w, yy], fill=(120, 110, 100), width=2)
    elif clase == "casa_rural":
        w, h = rng.uniform(0.32, 0.44) * S, rng.uniform(0.2, 0.28) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 200)))
        d.polygon([(x0 - 0.04 * S, y0), (x0 + w + 0.04 * S, y0), (x0 + w / 2, y0 - rng.uniform(0.14, 0.2) * S)],
                  fill=_color(rng, (170, 70, 50), 25))
        d.rectangle([x0 + w * 0.42, y0 + h * 0.4, x0 + w * 0.58, hor + 0.05 * S], fill=(110, 70, 40))
        d.rectangle([x0 + w * 0.1, y0 + h * 0.2, x0 + w * 0.3, y0 + h * 0.5], fill=(70, 100, 150))
    elif clase == "camping":
        d.rectangle([0, hor - 0.08 * S, S, S], fill=verde)
        w, h = rng.uniform(0.34, 0.5) * S, rng.uniform(0.28, 0.4) * S
        x0 = rng.uniform(0.05 * S, 0.95 * S - w); base = hor + 0.1 * S
        c = _color(rng, [(240, 140, 40), (60, 120, 200), (230, 200, 60)][int(rng.integers(0, 3))], 20)
        d.polygon([(x0, base), (x0 + w, base), (x0 + w / 2, base - h)], fill=c)
        d.polygon([(x0 + w * 0.4, base), (x0 + w * 0.6, base), (x0 + w / 2, base - h * 0.5)], fill=(50, 40, 40))
    if clase != "piscina" and rng.random() < 0.6:                              # árbol de adorno
        tx = rng.choice([rng.uniform(0.03, 0.12), rng.uniform(0.88, 0.97)]) * S
        d.rectangle([tx - 2, hor - 0.02 * S, tx + 2, hor + 0.12 * S], fill=(100, 70, 40))
        d.ellipse([tx - 0.07 * S, hor - 0.16 * S, tx + 0.07 * S, hor + 0.01 * S], fill=(50, 120, 60))
    return _final(img.resize((tam, tam), Image.LANCZOS), rng, ruido)


def dibujar_forma(clase, rng, tam=48):
    """Formas y texturas abstractas (tarea auxiliar de preentrenamiento). Devuelve tam×tam×3 uint8."""
    fondo, tinta = _color(rng, (rng.integers(20, 235),) * 3, 20), _color(rng, tuple(int(v) for v in rng.integers(0, 256, 3)), 10)
    if np.abs(np.array(fondo, dtype=int) - np.array(tinta, dtype=int)).sum() < 150:      # asegura contraste
        tinta = tuple(255 - c for c in fondo)
    E = 2
    S = tam * E
    img = Image.new("RGB", (S, S), fondo)
    d = ImageDraw.Draw(img)
    paso = int(rng.integers(6, 12)) * E
    off = int(rng.integers(0, paso))
    if clase == "rayas_horizontales":
        for y in range(off, S, paso): d.rectangle([0, y, S, y + paso // 2], fill=tinta)
    elif clase == "rayas_verticales":
        for x in range(off, S, paso): d.rectangle([x, 0, x + paso // 2, S], fill=tinta)
    elif clase == "diagonales":
        for k in range(-S, S, paso): d.line([k + off, 0, k + off + S, S], fill=tinta, width=max(2, paso // 3))
    elif clase == "cuadricula":
        for y in range(off, S, paso): d.line([0, y, S, y], fill=tinta, width=2)
        for x in range(off, S, paso): d.line([x, 0, x, S], fill=tinta, width=2)
    elif clase == "puntos":
        r = max(2, paso // 4)
        for y in range(off, S, paso):
            for x in range(off, S, paso): d.ellipse([x - r, y - r, x + r, y + r], fill=tinta)
    else:
        r = rng.uniform(0.18, 0.38) * S
        cx, cy = rng.uniform(r, S - r), rng.uniform(r, S - r)
        if clase == "circulo": d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=tinta)
        elif clase == "rectangulo": d.rectangle([cx - r, cy - r * rng.uniform(0.5, 1), cx + r, cy + r * rng.uniform(0.5, 1)], fill=tinta)
        else: d.polygon([(cx - r, cy + r), (cx + r, cy + r), (cx, cy - r)], fill=tinta)
    return _final(img.resize((tam, tam), Image.LANCZOS), rng)


def generar_lote(funcion, clases, n_por_clase, semilla=42, tam=48):
    """Devuelve (X uint8 [n, tam, tam, 3], y int [n]) con n_por_clase imágenes por clase, mezcladas."""
    rng = np.random.default_rng(semilla)
    X = np.stack([funcion(c, rng, tam) for c in clases for _ in range(n_por_clase)])
    y = np.repeat(np.arange(len(clases)), n_por_clase)
    orden = rng.permutation(len(y))
    return X[orden], y[orden]


PARECIDAS = {"hotel": "apartamentos", "apartamentos": "hotel", "casa_rural": "camping", "camping": "casa_rural", "piscina": "camping"}


def crear_dataset_carpetas(carpeta, n_por_clase=100, semilla=7, tam=48, ruido=8, prop_ambiguas=0.0):
    """Escribe PNG en carpeta/<clase>/img_000.png (estructura que espera image_dataset_from_directory).

    prop_ambiguas: fracción de imágenes cuyo dibujo corresponde a una clase parecida (PARECIDAS) pero se guarda
    en la carpeta de la otra: simula etiquetas dudosas o erróneas. Se anotan en carpeta/_ambiguas.csv (solo docente).
    """
    import os
    rng = np.random.default_rng(semilla)
    ambiguas = []
    for clase in CLASES_ALOJ:
        os.makedirs(os.path.join(carpeta, clase), exist_ok=True)
        for i in range(n_por_clase):
            dibujo = PARECIDAS[clase] if rng.random() < prop_ambiguas else clase
            if dibujo != clase:
                ambiguas.append(f"{clase}/img_{i:03d}.png,{dibujo}")
            Image.fromarray(dibujar_alojamiento(dibujo, rng, tam, ruido)).save(os.path.join(carpeta, clase, f"img_{i:03d}.png"))
    with open(os.path.join(carpeta, "_ambiguas.csv"), "w") as f:
        f.write("archivo,dibujo_real\n" + "\n".join(ambiguas) + "\n")
    return carpeta

In [ ]:
X_f, y_f = generar_lote(dibujar_forma, CLASES_FORMAS, 6, semilla=1)
X_a, y_a = generar_lote(dibujar_alojamiento, CLASES_ALOJ, 6, semilla=1)
fig, ejes = plt.subplots(2, 10, figsize=(14, 3.2))
for i in range(10):
    ejes[0, i].imshow(X_f[i]); ejes[0, i].set_title(CLASES_FORMAS[y_f[i]], fontsize=7); ejes[0, i].axis("off")
    ejes[1, i].imshow(X_a[i]); ejes[1, i].set_title(CLASES_ALOJ[y_a[i]], fontsize=7); ejes[1, i].axis("off")
plt.suptitle("Arriba: tarea auxiliar (formas y texturas) · Abajo: tarea real (tipos de alojamiento esquemáticos)")
plt.tight_layout(); plt.show()

## 1. La tarea real, con pocos datos

**Idea.** Imagina que TurisData solo tiene **6 imágenes por tipo de alojamiento** (30 en total) etiquetadas y con mucho ruido.
Es muy poco para una CNN desde cero. Reservamos 200 para validación y 300 para prueba.

In [ ]:
N_POR_CLASE = 6
alojamiento = functools.partial(dibujar_alojamiento, ruido=40)     # imágenes "sucias": mucho ruido, como fotos de mala calidad
X_ent, y_ent = generar_lote(alojamiento, CLASES_ALOJ, N_POR_CLASE, semilla=10)
X_val, y_val = generar_lote(alojamiento, CLASES_ALOJ, 40, semilla=11)
X_pru, y_pru = generar_lote(alojamiento, CLASES_ALOJ, 60, semilla=12)
print("entrenamiento", X_ent.shape, "· validación", X_val.shape, "· prueba", X_pru.shape)

## 2. Punto de partida: una CNN desde cero
La red se compone de una **base** (bloques `Conv → Pool` que extraen rasgos y terminan en
`Flatten`, que aplana los mapas en una lista de rasgos) y una **cabeza** (capa `Dense` que decide la clase).
Esta división es la clave del *transfer learning*: **la base se reutiliza; la cabeza se cambia**.

In [ ]:
def construir_base(nombre="base"):
    entrada = keras.layers.Input(shape=(48, 48, 3))
    x = keras.layers.Rescaling(1 / 255)(entrada)                     # la propia red normaliza los píxeles
    for i, f in enumerate((16, 32, 64), start=1):
        x = keras.layers.Conv2D(f, 3, padding="same", activation="relu", name=f"bloque{i}_conv")(x)
        x = keras.layers.MaxPooling2D(name=f"bloque{i}_pool")(x)
    x = keras.layers.Flatten(name="aplanar")(x)                       # 6×6×64 = 2304 rasgos
    return keras.Model(entrada, x, name=nombre)

def con_cabeza(base, n_clases, dropout=0.2, tasa=1e-3):
    """Base + cabeza nueva (Dropout y Dense softmax), ya compilado."""
    entrada = keras.layers.Input(shape=(48, 48, 3))
    x = base(entrada)
    x = keras.layers.Dropout(dropout)(x)
    salida = keras.layers.Dense(n_clases, activation="softmax", name="cabeza")(x)
    modelo = keras.Model(entrada, salida)
    modelo.compile(keras.optimizers.Adam(tasa), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

def entrenar(modelo, epocas=30, semilla=SEMILLA, X=None, y=None):
    keras.utils.set_random_seed(semilla)
    X = X_ent if X is None else X
    y = y_ent if y is None else y
    return modelo.fit(X, y, validation_data=(X_val, y_val), epochs=epocas, batch_size=16, verbose=0)

keras.utils.set_random_seed(SEMILLA)
desde_cero = con_cabeza(construir_base("base_desde_cero"), 5)
h_cero = entrenar(desde_cero, epocas=40)
acc_cero = desde_cero.evaluate(X_pru, y_pru, verbose=0)[1]
print(f"CNN desde cero con {len(X_ent)} imágenes → precisión en prueba: {acc_cero:.3f}")

## 3. Preentrenar la base en una tarea auxiliar

En la vida real se descargaría una base entrenada con **millones de fotos** (ImageNet). Aquí, para poder trabajar sin
internet, la base aprende primero una tarea **distinta pero relacionada**: reconocer 8 clases de formas y
texturas (rayas, cuadrículas, círculos, triángulos…) con 3200 imágenes. Esos rasgos (bordes, esquinas, patrones
repetidos) también sirven para reconocer edificios y tiendas de campaña.

In [ ]:
X_aux, y_aux = generar_lote(dibujar_forma, CLASES_FORMAS, 400, semilla=3)
X_aux_v, y_aux_v = generar_lote(dibujar_forma, CLASES_FORMAS, 50, semilla=4)

keras.utils.set_random_seed(SEMILLA)
base_pre = construir_base("base_preentrenada")
modelo_aux = con_cabeza(base_pre, len(CLASES_FORMAS), dropout=0.0)
modelo_aux.fit(X_aux, y_aux, epochs=12, batch_size=64, verbose=0)
print("Precisión en la tarea auxiliar (validación):", round(modelo_aux.evaluate(X_aux_v, y_aux_v, verbose=0)[1], 3))
print("La base tiene", base_pre.count_params(), "parámetros y ya sabe extraer rasgos visuales.")

## 4. Congelar la base y entrenar solo una cabeza nueva

**Idea.** "Congelar" una capa (`trainable = False`) significa que su peso **no se actualiza** al entrenar. Así
conservamos lo aprendido y solo se entrenan las pocas neuronas de la cabeza: hacen falta menos datos y menos tiempo.

**Ejercicio 1.** Escribe `congelar(modelo)` que ponga `trainable = False` en **todas** las capas del modelo dado
(es decir, `modelo.trainable = False`) y lo devuelva. Después crea `copia_base` (una copia independiente de
`base_pre`, ya lo tienes hecho abajo) y congélala.

In [ ]:
# TODO: modelo.trainable = False y return modelo
...

def copiar_base(base, nombre):
    """Copia independiente de la base con los mismos pesos (para no modificar la original)."""
    copia = keras.models.clone_model(base)
    copia.set_weights(base.get_weights())
    copia._name = nombre
    return copia

copia_base = congelar(copiar_base(base_pre, "base_congelada"))
assert len(copia_base.trainable_weights) == 0, "Tras congelar no debe quedar ningún peso entrenable en la base"
assert len(copia_base.non_trainable_weights) == len(base_pre.weights), "Todos los pesos de la base deben pasar a 'no entrenables'"
assert len(base_pre.trainable_weights) > 0, "La base original no debe quedar congelada (usa una copia)"

In [ ]:
transfer = con_cabeza(copia_base, 5)
n_entrenables = sum(int(np.prod(p.shape)) for p in transfer.trainable_weights)
print(f"Parámetros totales: {transfer.count_params()} · entrenables (solo la cabeza): {n_entrenables}")
h_transfer = entrenar(transfer, epocas=30)
acc_transfer = transfer.evaluate(X_pru, y_pru, verbose=0)[1]
print(f"Transfer learning (base congelada) → precisión en prueba: {acc_transfer:.3f}")

## 5. *Fine-tuning*: descongelar las últimas capas con cuidado

**Idea.** Una vez entrenada la cabeza, se puede **descongelar** el último bloque de la base y seguir entrenando con una
**tasa de aprendizaje muy pequeña** (10 veces menor o más): se ajustan los rasgos más específicos sin destrozar los generales.
Regla de oro: *primero cabeza, después fine-tuning, y con tasa pequeña*. Los primeros bloques (bordes, texturas) son
universales y casi nunca se tocan.

**Ejercicio 2.** Escribe `descongelar_bloque(base, n)` que ponga `trainable = True` **solo** en las capas de la base cuyo
nombre empieza por `f"bloque{n}"` (por ejemplo `bloque3_conv`) y devuelva el número de pesos entrenables resultantes
(`len(base.trainable_weights)`). Antes, `base.trainable` debe estar en `True` para poder cambiar capas sueltas,
así que pon ese valor y después congela a mano las demás capas.

In [ ]:
# TODO: base.trainable = True; recorre base.layers y pon trainable según el nombre de la capa
...

_prueba = copiar_base(base_pre, "prueba")
assert descongelar_bloque(_prueba, 3) == 2, "El bloque 3 tiene una convolución con 2 pesos entrenables (filtros y sesgos)"
assert not _prueba.get_layer("bloque1_conv").trainable, "Los bloques 1 y 2 deben seguir congelados"
assert descongelar_bloque(_prueba, 2) == 2 and not _prueba.get_layer("bloque3_conv").trainable, "Solo debe estar activo el bloque indicado"

In [ ]:
# Fine-tuning: partimos del modelo transfer ya entrenado (cabeza lista), descongelamos el bloque 3 y bajamos la tasa
n_ent_ft = descongelar_bloque(copia_base, 3)
transfer.compile(keras.optimizers.Adam(1e-4), "sparse_categorical_crossentropy", metrics=["accuracy"])   # ¡recompilar!
h_ft = entrenar(transfer, epocas=25, semilla=1)
acc_ft = transfer.evaluate(X_pru, y_pru, verbose=0)[1]
print(f"Tras fine-tuning del bloque 3 → precisión en prueba: {acc_ft:.3f}")

### Comparación con varias semillas
Un solo entrenamiento con 100 imágenes es muy variable. Repetimos las tres estrategias con **3 semillas** (la base
preentrenada es la misma; cambia la inicialización de la cabeza y el orden de los lotes).

In [ ]:
def experimento(semilla, X=None, y=None, epocas=30):
    """Devuelve la precisión en prueba de (desde cero, base congelada, base con fine-tuning)."""
    keras.utils.set_random_seed(semilla)
    m0 = con_cabeza(construir_base("cero"), 5)
    entrenar(m0, epocas + 10, semilla, X, y)
    b = congelar(copiar_base(base_pre, "b"))
    m1 = con_cabeza(b, 5)
    entrenar(m1, epocas, semilla, X, y)
    a1 = m1.evaluate(X_pru, y_pru, verbose=0)[1]
    descongelar_bloque(b, 3)
    m1.compile(keras.optimizers.Adam(1e-4), "sparse_categorical_crossentropy", metrics=["accuracy"])
    entrenar(m1, epocas - 5, semilla, X, y)
    return m0.evaluate(X_pru, y_pru, verbose=0)[1], a1, m1.evaluate(X_pru, y_pru, verbose=0)[1]

filas = [experimento(s) for s in (1, 2, 3)]
comparacion = pd.DataFrame(filas, columns=["Desde cero", "Base congelada", "Fine-tuning bloque 3"]).agg(["mean", "std"]).round(3)
comparacion

In [ ]:
comparacion.loc["mean"].plot(kind="bar", yerr=comparacion.loc["std"], capsize=4, color=["gray", "steelblue", "seagreen"])
plt.ylabel("precisión en prueba"); plt.ylim(0, 1); plt.xticks(rotation=0)
plt.title(f"Con solo {N_POR_CLASE} imágenes por clase (media de 3 semillas)"); plt.show()

**Cómo leerlo.** Con pocas imágenes, la base preentrenada suele dar mejor punto de partida que aprender todo desde cero,
y el *fine-tuning* añade un pequeño ajuste (o no: con muy pocos datos puede sobreajustar). Los resultados exactos varían;
lo importante es el método. **Con pesos reales de ImageNet la diferencia es mucho mayor**, porque esa base ha visto
millones de fotos de objetos reales, no 3 200 dibujos.

## 6. Con pesos reales: MobileNetV2 (rama Colab) y qué cambia

**Qué cambia con pesos reales de ImageNet:**

| Aspecto | Rama offline (este notebook) | Con MobileNetV2 + ImageNet |
|---|---|---|
| Base | 3 bloques, ≈ 23 000 parámetros, preentrenada en 3 200 dibujos | ≈ 2,2 millones de parámetros, entrenada con 1,2 M de fotos |
| Tamaño de entrada | 48×48 | 96, 128, 160 o 224 (mínimo 32; con 224 es la configuración estándar) |
| Preprocesado | `Rescaling(1/255)` dentro del modelo | `keras.applications.mobilenet_v2.preprocess_input` (píxeles a [−1, 1]) |
| Normalización | sin `BatchNormalization` | la base lleva *batch norm*: al congelarla se llama con `training=False` |
| Resultado | mejora modesta | suele bastar con 20–100 fotos por clase para resultados útiles |

La **estructura no cambia**: base congelada → cabeza nueva → fine-tuning con tasa pequeña.

In [ ]:
# Esta celda funciona SIN internet: construye MobileNetV2 con pesos aleatorios (weights=None) para ver su estructura.
def construir_mobilenet(pesos=None, tam=96):
    """pesos=None (aleatorios, offline) o pesos='imagenet' (requiere internet, se descarga ≈ 10 MB)."""
    base = keras.applications.MobileNetV2(input_shape=(tam, tam, 3), include_top=False, weights=pesos, alpha=0.35)
    base.trainable = False
    entrada = keras.layers.Input(shape=(48, 48, 3))
    x = keras.layers.Resizing(tam, tam)(entrada)                                     # de 48×48 a tam×tam
    x = keras.applications.mobilenet_v2.preprocess_input(x)                          # píxeles → [−1, 1]
    x = base(x, training=False)                                                      # ¡training=False al congelar!
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.2)(x)
    salida = keras.layers.Dense(5, activation="softmax")(x)
    modelo = keras.Model(entrada, salida)
    modelo.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo, base

mn, mn_base = construir_mobilenet(pesos=None)
print("MobileNetV2 (alpha 0.35): capas de la base =", len(mn_base.layers), "· parámetros totales =", mn.count_params(),
      "· entrenables =", sum(int(np.prod(p.shape)) for p in mn.trainable_weights))
print("Salida del modelo para 2 imágenes sintéticas:", mn.predict(X_ent[:2], verbose=0).round(2))

> **AVISO: la siguiente celda requiere internet y NO se ha verificado en el taller.** Descarga los pesos de ImageNet de
> MobileNetV2 (≈ 10 MB para `alpha=0.35`) y entrena la cabeza sobre las imágenes sintéticas de este notebook (solo para ver
> la mecánica; el reto usará tus fotos). Pon `USAR_INTERNET = True` en Colab.

In [ ]:
USAR_INTERNET = False        # ← True solo con conexión (Colab)

if USAR_INTERNET:
    mn_real, mn_real_base = construir_mobilenet(pesos="imagenet")           # ← aquí se descargan los pesos
    mn_real.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=15, batch_size=16, verbose=0)
    print("MobileNetV2 con ImageNet, precisión en prueba:", round(mn_real.evaluate(X_pru, y_pru, verbose=0)[1], 3))
    # Fine-tuning: se descongelan las últimas capas de la base y se baja la tasa (recompilar siempre)
    mn_real_base.trainable = True
    for capa in mn_real_base.layers[:-20]:
        capa.trainable = False
    mn_real.compile(keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
    mn_real.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=10, batch_size=16, verbose=0)
    print("Tras fine-tuning:", round(mn_real.evaluate(X_pru, y_pru, verbose=0)[1], 3))
else:
    print("Rama de Colab desactivada (USAR_INTERNET = False).")

## 7. Mini-reto integrador: ¿cuántas imágenes necesito?

El cliente pregunta: *"¿Cuántas fotos por tipo de alojamiento tenemos que etiquetar?"* Responde con datos:
entrena la **CNN desde cero** y el **transfer learning con base congelada** con
`n` ∈ {3, 6, 12, 24} imágenes por clase (genéralas con `generar_lote(alojamiento, CLASES_ALOJ, n, semilla=20)`)
y guarda en el diccionario `curva_datos` la precisión en prueba de cada estrategia, con la forma
`{n: (acc_desde_cero, acc_transfer)}`. Usa 40 épocas para "desde cero" y 30 para "transfer".

In [ ]:
# TODO: Para cada n: genera datos, entrena ambos modelos con entrenar(modelo, epocas, semilla, X, y) y evalúa en X_pru
...

assert set(curva_datos) == {3, 6, 12, 24}, "Debe haber una entrada por cada n: 3, 6, 12 y 24"
assert all(len(v) == 2 and 0 <= v[0] <= 1 and 0 <= v[1] <= 1 for v in curva_datos.values()), "Cada valor debe ser (precisión desde cero, precisión transfer) entre 0 y 1"

ns = sorted(curva_datos)
plt.plot(ns, [curva_datos[n][0] for n in ns], "o-", label="desde cero")
plt.plot(ns, [curva_datos[n][1] for n in ns], "s-", label="transfer learning (base congelada)")
plt.xlabel("imágenes por clase"); plt.ylabel("precisión en prueba"); plt.ylim(0, 1.02)
plt.title("¿Cuántas imágenes hacen falta?"); plt.legend(); plt.show()

**Tu recomendación al cliente (edita esta celda):** …

## 8. Preguntas de reflexión
1. ¿Por qué se cambia la cabeza pero se reutiliza la base?
2. ¿Por qué el *fine-tuning* se hace con una tasa de aprendizaje mucho más pequeña y **después** de entrenar la cabeza?
3. Si tus fotos son muy distintas de las de ImageNet (por ejemplo, imágenes térmicas o de satélite), ¿el *transfer learning* seguirá ayudando? ¿Y descongelar más capas?
4. ¿Qué riesgo hay en usar un modelo preentrenado que no has entrenado tú (sesgos, licencias, datos con los que se entrenó)?

## 9. Resumen: qué has aprendido
- *Transfer learning* = reutilizar la **base** de una red preentrenada y entrenar una **cabeza** nueva.
- Se congela (`trainable = False`), se entrena la cabeza y, si hace falta, se hace *fine-tuning* de las últimas capas con tasa baja (y recompilando).
- Ayuda sobre todo con **pocos datos**; su ventaja depende de cuánto se parezcan las tareas.
- Con MobileNetV2/ImageNet cambian el preprocesado, el tamaño de entrada y el uso de `training=False`, pero no la estructura.